Bu kod, data/normal ve data/layers klasörlerini tarar, tüm eşleşen görüntü çiftlerini sırasıyla işler ve cropped_output/ klasörüne düzgün dosya adlarıyla kaydeder (output_normal_1.png, output_layer_1.png, ...).

In [4]:
# OCT Normal Image Crop Pipeline (only normal images)
# - Crops: TOP (height), LEFT (width start), RIGHT (width end)
# - Saves as: cropped_output/output_normal_#.png

import os
import cv2

# =========================
# 1) Crop Ayarları (PARAMS)
# =========================
# TOP: görüntünün üst kısmından ne kadarını alacağız?
#   top = int(h / top_div)
#   top_div büyüdükçe -> daha az yükseklik alınır (daha fazla üstten kesilmiş olur)
TOP_DIV = 1.30

# LEFT: soldan ne kadar kesilecek?
#   left = int(w / left_div)
#   left_div küçüldükçe -> left büyür -> soldan daha fazla keser
LEFT_DIV = 2.10

# RIGHT: sağdan ne kadar kesilecek?
#   right = int(w * right_ratio)
#   right_ratio küçüldükçe -> sağdan daha fazla keser
RIGHT_RATIO = 0.93


# =========================
# 2) Crop Fonksiyonu
# =========================
def crop_normal_oct(normal_img, top_div=TOP_DIV, left_div=LEFT_DIV, right_ratio=RIGHT_RATIO):
    """
    normal_img üzerinde:
      - üstten crop: y = 0 .. top
      - soldan crop: x = left .. right
      - sağdan crop: x = right'tan sonrası atılır
    """
    if normal_img is None:
        return None

    h, w = normal_img.shape[:2]

    # --- ÜSTTEN KESME ---
    # y_end = top, yani sadece 0..top arası kalır (top'tan sonrası atılır)
    top = int(h / top_div)

    # --- SOLDAN KESME ---
    # x_start = left, yani 0..left arası atılır
    left = int(w / left_div)

    # --- SAĞDAN KESME ---
    # x_end = right, yani right..w arası atılır
    right = int(w * right_ratio)

    # Güvenlik: right, left'ten küçük olmasın
    if right <= left:
        right = w  # saçma bir durumda sağ sınırı full yap

    # Crop uygula: image[y_start:y_end, x_start:x_end]
    cropped = normal_img[:top, left:right]
    return cropped


# =========================
# 3) Klasörler
# =========================
input_dir = "data/normal"
output_dir = "cropped_output"
os.makedirs(output_dir, exist_ok=True)

# =========================
# 4) Dosyaları Listele
# =========================
image_files = sorted([
    f for f in os.listdir(input_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"))
])

print(f"Toplam {len(image_files)} görüntü bulundu.\n")

# =========================
# 5) Loop: Oku -> Crop -> Kaydet
# =========================
for idx, img_name in enumerate(image_files, 1):
    img_path = os.path.join(input_dir, img_name)

    # IMREAD_UNCHANGED: 8-bit/16-bit vs ne varsa bozmadan okur
    normal_img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)

    if normal_img is None:
        print(f"[{idx}] {img_name} okunamadı, atlanıyor.")
        continue

    cropped = crop_normal_oct(normal_img)

    out_path = os.path.join(output_dir, f"output_normal_{idx}.png")

    # PNG compression 0: en az sıkıştırma (daha hızlı / daha büyük dosya)
    cv2.imwrite(out_path, cropped, [cv2.IMWRITE_PNG_COMPRESSION, 0])

    # Debug: hangi sınırlar ile kestiğini görmek istersen:
    h, w = normal_img.shape[:2]
    top_dbg = int(h / TOP_DIV)
    left_dbg = int(w / LEFT_DIV)
    right_dbg = int(w * RIGHT_RATIO)
    print(f"[{idx}] {img_name} -> {out_path}")
    print(f"     ÜSTTEN kesim: y=0..{top_dbg} (h={h})")
    print(f"     SOLDAN kesim: x=0..{left_dbg} atıldı (w={w})")
    print(f"     SAĞDAN kesim: x={right_dbg}..{w} atıldı\n")

print("Tüm normal görüntüler kırpıldı.")

Toplam 53 görüntü bulundu.

[1] WhatsApp Image 2026-06-02 at 11.25.38 (2).jpeg -> cropped_output/output_normal_1.png
     ÜSTTEN kesim: y=0..458 (h=596)
     SOLDAN kesim: x=0..601 atıldı (w=1264)
     SAĞDAN kesim: x=1175..1264 atıldı

[2] WhatsApp Image 2026-06-02 at 11.25.38.jpeg -> cropped_output/output_normal_2.png
     ÜSTTEN kesim: y=0..458 (h=596)
     SOLDAN kesim: x=0..601 atıldı (w=1264)
     SAĞDAN kesim: x=1175..1264 atıldı

[3] WhatsApp Image 2026-06-02 at 11.25.39 (1).jpeg -> cropped_output/output_normal_3.png
     ÜSTTEN kesim: y=0..458 (h=596)
     SOLDAN kesim: x=0..601 atıldı (w=1264)
     SAĞDAN kesim: x=1175..1264 atıldı

[4] WhatsApp Image 2026-06-02 at 11.25.39 (2).jpeg -> cropped_output/output_normal_4.png
     ÜSTTEN kesim: y=0..458 (h=596)
     SOLDAN kesim: x=0..601 atıldı (w=1264)
     SAĞDAN kesim: x=1175..1264 atıldı

[5] WhatsApp Image 2026-06-02 at 11.25.39 (3).jpeg -> cropped_output/output_normal_5.png
     ÜSTTEN kesim: y=0..458 (h=596)
     SOLDAN kes